In [ ]:
 
import json
import linecache 
import pandas as pd
import time 
import numpy as np
import matplotlib.pyplot as plt





In [ ]:
 
class CsvConverter:
    def __init__(self, csvfile, jsonfile):
        self.csvfile = csvfile
        self.jsonfile = jsonfile
        self.keys = self.header_maker()
        self.data = self.data_maker()
    
    
    def header_maker(self):
        header_line = linecache.getline(self.csvfile, 1)
        keys = header_line.strip().split(',')
        return keys
 
    def data_maker(self):
 
        data = []
        rows = len(linecache.getlines(self.csvfile))
        for i in range(2, rows+1):
            row_line = linecache.getline(self.csvfile, i)
            row_values = row_line.strip().split(',')
            assert len(row_values) == len(self.keys), f"Warning: Skipping line {i-1} as number of elements don't match number of keys in header"
            row_dict = dict(zip(self.keys, row_values))
            data.append(row_dict)
    
        return data    

 
    def csv_to_json(self):
 
        json_data = {'data': self.data, 'header': self.keys}
        with open(self.jsonfile, 'w') as json_file:
            json.dump(json_data, json_file, indent=4)
        


In [ ]:
converter = CsvConverter('dSST.csv', 'output.json') 

In [ ]:
class Reader:
    def __init__(self, csvfile, stride):
        self.csvfile = csvfile
        self.stride = stride
        self.pointer = 0
        self.converter = CsvConverter(self.csvfile, 'foo.json')
        self.data = self.converter.data
        self.keys = self.converter.keys
        self.observers = set()
        
    def add_observer(self, observer):
        self.observers.add(observer)

    def remove_observer(self, observer):
        self.observers.discard(observer)

    def notify_observers(self):
        for observer in self.observers:
            observer.update()

    def get_lines(self):
        if self.pointer >= len(self.data):
            return ''
        lines = self.data[self.pointer:self.pointer+self.stride]
        self.pointer += self.stride
        json_data = {'data': lines}
        time.sleep(5) 
        self.notify_observers()
        return json.dumps(json_data, indent = 4)

In [ ]:
reader = Reader('dSST.csv', 5)
while True:
    lines = reader.get_lines()
    if not lines:
        break
    print(lines)


In [ ]:
 
class AverageYear:
    def __init__(self, csvfile, stride):
        self.reader = Reader(csvfile, stride)
        self.data = pd.DataFrame()
        self.fig, self.ax = plt.subplots()
        self.reader.add_observer(self)
    
    def calculate_average(self):
        while True:
            lines = self.reader.get_lines()
            if not lines:
                break
            json_data = json.loads(lines)
            df = pd.DataFrame(json_data['data'])
            df = df.astype({'Year': int})
            df = df.set_index('Year')
            self.data = self.data.append(df)
            
        # Calculate average temperature anomaly for all years
        self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']] = self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']].apply(pd.to_numeric)
        self.data['Yearly Average'] = self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']].mean(axis=1)
        
        # Plot the average temperature anomaly for all years
        self.ax.plot(self.data.index, self.data['Yearly Average'])
        self.ax.set_xlabel('Year')
        self.ax.set_ylabel('Average Temperature Anomaly')
        self.ax.set_title('Average Temperature Anomaly for All Years')
        plt.show()

    def update(self):
        self.calculate_average()    


In [ ]:
average_year = AverageYear('dSST.csv',5)
average_year.calculate_average()

In [ ]:
 
class AverageMonth:
    def __init__(self, csvfile, stride):
        self.reader = Reader(csvfile, stride)
        self.data = pd.DataFrame()
        self.fig, self.ax = plt.subplots()
        self.months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        self.reader.add_observer(self)
    
    def calculate_average(self):
        while True:
            lines = self.reader.get_lines()
            if not lines:
                break
            json_data = json.loads(lines)
            df = pd.DataFrame(json_data['data'])
            df = df.astype({'Year': int})
            df = df[self.months] 
            self.data = self.data.append(df)
         
        self.data = self.data.apply(pd.to_numeric)
        self.monthly_averages = self.data.mean(axis=0)
 
        self.ax.plot(self.months, self.monthly_averages)
        self.ax.set_xlabel('Month')
        self.ax.set_ylabel('Average Monthly Temperature Anomaly')
        self.ax.set_title('Average Monthly Temperature Anomaly for All Years')
        plt.show()
        
    def update(self):
        self.calculate_average()    


In [ ]:
average_month = AverageMonth('dSST.csv', 5)
average_month.calculate_average()
